# nuScenes 리더보드 제출 파이프라인

## 목표
LSS BEV 모델의 출력을 nuScenes 공식 제출 포맷으로 변환하고,
nuscenes-devkit으로 공식 메트릭(NDS, mAP)을 계산한다.

## nuScenes 평가 메트릭
| 메트릭 | 설명 |
|---|---|
| **NDS** | nuScenes Detection Score (최종 종합 점수) |
| **mAP** | 10개 클래스 평균 Average Precision |
| mATE | 위치 오차 (Translation Error, m) |
| mASE | 크기 오차 (Scale Error, 1-IoU) |
| mAOE | 방향 오차 (Orientation Error, rad) |
| mAVE | 속도 오차 (Velocity Error, m/s) |
| mAAE | 속성 오차 (Attribute Error) |

## 전략
nuScenes mini(4GB)가 없어도 동작하도록 2단계로 구성:
1. **Mock 데이터**: nuScenes 스키마에 맞는 예측 생성 → 포맷/평가 파이프라인 검증
2. **실제 데이터**: mini 다운로드 경로 제공 → 동일 파이프라인 재사용

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

## 셀 1: nuscenes-devkit 설치 및 확인

In [ ]:
import subprocess, sys

try:
    import nuscenes
    print("nuscenes-devkit 이미 설치됨")
except ImportError:
    print("nuscenes-devkit==1.0.5 설치 중...")
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "nuscenes-devkit==1.0.5", "-q"])
    print("설치 완료")

# numpy 2.x 호환성 패치 (nuscenes-devkit 1.0.5는 np.float 사용)
import numpy as np
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "bool"):
    np.bool = bool
if not hasattr(np, "complex"):
    np.complex = complex

import json, uuid
from pathlib import Path
import matplotlib.pyplot as plt
from pyquaternion import Quaternion

print("필요 패키지 로드 완료")

## 셀 2: nuScenes 제출 포맷 이해

### 공식 제출 JSON 스키마
```json
{
  "meta": {
    "use_camera": true,
    "use_lidar": false
  },
  "results": {
    "<sample_token>": [
      {
        "sample_token": "abc123...",
        "translation": [x, y, z],
        "size": [width, length, height],
        "rotation": [w, x, y, z],
        "velocity": [vx, vy],
        "detection_name": "car",
        "detection_score": 0.85,
        "attribute_name": "vehicle.moving"
      }
    ]
  }
}
```

### 10개 감지 클래스
car, truck, bus, trailer, construction_vehicle,
pedestrian, motorcycle, bicycle, traffic_cone, barrier

In [ ]:
DETECTION_NAMES = [
    'car', 'truck', 'bus', 'trailer', 'construction_vehicle',
    'pedestrian', 'motorcycle', 'bicycle', 'traffic_cone', 'barrier'
]

DEFAULT_SIZES = {
    'car':                  [1.93, 4.63, 1.72],
    'truck':                [2.51, 6.93, 2.84],
    'bus':                  [2.96, 11.19, 3.49],
    'trailer':              [2.90, 12.29, 3.87],
    'construction_vehicle': [2.63, 6.37, 3.19],
    'pedestrian':           [0.77, 0.73, 1.76],
    'motorcycle':           [0.96, 2.11, 1.47],
    'bicycle':              [0.60, 1.70, 1.28],
    'traffic_cone':         [0.48, 0.48, 1.01],
    'barrier':              [2.53, 0.50, 0.98],
}

ATTR_TABLE = {
    'car':          ['vehicle.moving', 'vehicle.stopped', 'vehicle.parked'],
    'truck':        ['vehicle.moving', 'vehicle.stopped', 'vehicle.parked'],
    'bus':          ['vehicle.moving', 'vehicle.stopped', 'vehicle.parked'],
    'trailer':      ['vehicle.moving', 'vehicle.stopped', 'vehicle.parked'],
    'construction_vehicle': ['vehicle.moving', 'vehicle.stopped'],
    'pedestrian':   ['pedestrian.moving', 'pedestrian.standing', 'pedestrian.sitting_lying_down'],
    'motorcycle':   ['cycle.with_rider', 'cycle.without_rider'],
    'bicycle':      ['cycle.with_rider', 'cycle.without_rider'],
    'traffic_cone': [''],
    'barrier':      [''],
}

print('nuScenes 클래스 정의 완료')
print(f'감지 클래스 ({len(DETECTION_NAMES)}개):', ', '.join(DETECTION_NAMES))

## 셀 3: Mock 예측 생성 — nuScenes 스키마 준수

In [ ]:
np.random.seed(42)
N_SCENES  = 10
N_SAMPLES = 4
N_DET_MAX = 15

def make_token():
    return uuid.uuid4().hex

def velocity_to_attr(det_name, vx, vy):
    speed = np.sqrt(vx**2 + vy**2)
    attrs = ATTR_TABLE[det_name]
    if '' in attrs:
        return ''
    if det_name in ['car','truck','bus','trailer','construction_vehicle']:
        if speed > 0.5:   return 'vehicle.moving'
        elif speed > 0.1: return 'vehicle.stopped'
        else:             return 'vehicle.parked'
    elif det_name == 'pedestrian':
        if speed > 0.5:   return 'pedestrian.moving'
        elif speed > 0.1: return 'pedestrian.standing'
        else:             return 'pedestrian.sitting_lying_down'
    else:
        return 'cycle.with_rider' if speed > 0.5 else 'cycle.without_rider'

submission = {
    "meta": {
        "use_camera": True, "use_lidar": False,
        "use_radar": False, "use_map": False, "use_external": False
    },
    "results": {}
}

sample_tokens = []

for scene_i in range(N_SCENES):
    for sample_i in range(N_SAMPLES):
        token = make_token()
        sample_tokens.append(token)
        n_det = np.random.randint(3, N_DET_MAX)
        preds = []
        for _ in range(n_det):
            det_name = np.random.choice(DETECTION_NAMES,
                p=[0.35,0.12,0.05,0.03,0.03,0.25,0.06,0.05,0.03,0.03])
            x, y, z = np.random.uniform(-50,50), np.random.uniform(-50,50), 0.0
            w, l, h = DEFAULT_SIZES[det_name]
            size = [w*np.random.uniform(0.9,1.1), l*np.random.uniform(0.9,1.1), h*np.random.uniform(0.9,1.1)]
            yaw = np.random.uniform(-np.pi, np.pi)
            q   = Quaternion(axis=[0,0,1], angle=yaw)
            speed = np.random.exponential(2.0)
            vx, vy = speed*np.cos(yaw), speed*np.sin(yaw)
            score = float(np.clip(np.random.beta(5,2), 0.25, 1.0))
            preds.append({
                "sample_token":    token,
                "translation":     [x, y, z],
                "size":            size,
                "rotation":        [q.w, q.x, q.y, q.z],
                "velocity":        [vx, vy],
                "detection_name":  det_name,
                "detection_score": score,
                "attribute_name":  velocity_to_attr(det_name, vx, vy),
            })
        submission['results'][token] = preds

total = sum(len(v) for v in submission['results'].values())
print(f'Mock 예측 생성 완료')
print(f'  총 샘플: {len(sample_tokens)}  |  총 박스: {total}  |  샘플당 평균: {total/len(sample_tokens):.1f}')

## 셀 4: 제출 JSON 저장 및 스키마 검증

In [ ]:
OUTPUT_DIR = Path('submission_output')
OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / 'results_nusc.json'

with open(SUBMISSION_PATH, 'w') as f:
    json.dump(submission, f, indent=2)

size_mb = SUBMISSION_PATH.stat().st_size / 1024 / 1024
print(f'저장 완료: {SUBMISSION_PATH} ({size_mb:.2f} MB)')

REQUIRED = ['sample_token','translation','size','rotation',
            'velocity','detection_name','detection_score','attribute_name']
VALID_CLS = set(DETECTION_NAMES)
errors = []
for token, preds in submission['results'].items():
    for p in preds:
        for f in REQUIRED:
            if f not in p: errors.append(f'{token[:8]}: {f} 누락')
        if p.get('detection_name') not in VALID_CLS:
            errors.append(f'잘못된 클래스: {p.get("detection_name")}')
        if not (0.0 <= p.get('detection_score',-1) <= 1.0):
            errors.append(f'신뢰도 범위 오류')

if errors:
    print(f'검증 실패 ({len(errors)}건):', errors[:3])
else:
    print('스키마 검증 통과 — 모든 필드 정상')

print('\n예시 박스:')
ex = submission['results'][sample_tokens[0]][0]
for k, v in ex.items(): print(f'  {k:<20}: {v}')

## 셀 5: 공식 nuScenes 메트릭 계산

In [ ]:
from nuscenes.eval.detection.config import config_factory
from nuscenes.eval.detection.data_classes import DetectionBox, DetectionMetricDataList
from nuscenes.eval.common.data_classes import EvalBoxes
from nuscenes.eval.detection.algo import accumulate, calc_ap, calc_tp

cfg = config_factory('detection_cvpr_2019')

print('평가 설정:')
print(f'  매칭 임계값: {cfg.dist_ths} m')
print(f'  평가 클래스: {cfg.class_names}')

def to_eval_boxes(results_dict):
    boxes = EvalBoxes()
    for token, preds in results_dict.items():
        box_list = []
        for p in preds:
            box_list.append(DetectionBox(
                sample_token    = token,
                translation     = tuple(p['translation']),
                size            = tuple(p['size']),
                rotation        = tuple(p['rotation']),
                velocity        = tuple(p['velocity']),
                detection_name  = p['detection_name'],
                detection_score = p['detection_score'],
                attribute_name  = p['attribute_name'],
            ))
        boxes.add_boxes(token, box_list)
    return boxes

# GT = 예측 상위 절반 (포맷 검증용; 실제 평가는 nuScenes DB 사용)
gt_dict = {}
for token, preds in submission['results'].items():
    gt_list = []
    for p in preds[:max(1, len(preds)//2)]:
        gt = dict(p); gt['detection_score'] = -1.0
        gt_list.append(gt)
    gt_dict[token] = gt_list

pred_boxes = to_eval_boxes(submission['results'])
gt_boxes   = to_eval_boxes(gt_dict)
print(f'예측 박스: {len(pred_boxes.all)} | GT 박스: {len(gt_boxes.all)}')

In [ ]:
metric_data_list = DetectionMetricDataList()
ap_results = {}

print('클래스별 AP 계산 중...')
for class_name in cfg.class_names:
    for dist_th in cfg.dist_ths:
        md_data = accumulate(gt_boxes, pred_boxes, class_name, cfg.dist_fcn_callable, dist_th)
        metric_data_list.set(class_name, dist_th, md_data)
    ap = np.mean([
        calc_ap(metric_data_list[(class_name, dt)], cfg.min_recall, cfg.min_precision)
        for dt in cfg.dist_ths
    ])
    ap_results[class_name] = ap

mAP = np.mean(list(ap_results.values()))

print(f'\n{"클래스":<25} {"AP":>8}')
print('-' * 35)
for cls, ap in sorted(ap_results.items(), key=lambda x: -x[1]):
    bar = chr(9608) * int(ap * 20)
    print(f'{cls:<25} {ap:>6.3f}  {bar}')
print('-' * 35)
print(f'{"mAP":<25} {mAP:>6.3f}')

In [ ]:
TP_NAMES  = ['trans_err','scale_err','orient_err','vel_err','attr_err']
TP_LABELS = {'trans_err':'mATE','scale_err':'mASE','orient_err':'mAOE','vel_err':'mAVE','attr_err':'mAAE'}
tp_results = {tp: [] for tp in TP_NAMES}

for class_name in cfg.class_names:
    md_data = metric_data_list[(class_name, min(cfg.dist_ths))]
    for tp_name in TP_NAMES:
        tp_results[tp_name].append(calc_tp(md_data, cfg.min_recall, tp_name))

mean_tp = {tp: np.nanmean(vals) for tp, vals in tp_results.items()}

# NDS = (5*mAP + sum(w*(1-min(1,TP)))) / (5 + sum(weights))
tp_weights = {'trans_err':1.0,'scale_err':1.0,'orient_err':1.0,'vel_err':1.0,'attr_err':0.5}
nds_num = 5 * mAP + sum(w * max(0, 1-min(1, mean_tp[tp])) for tp, w in tp_weights.items())
NDS = nds_num / (5 + sum(tp_weights.values()))

print('=' * 50)
print('  nuScenes 공식 메트릭 - 최종 결과')
print('=' * 50)
print(f'  NDS  : {NDS:.4f}')
print(f'  mAP  : {mAP:.4f}')
for tp_name, label in TP_LABELS.items():
    print(f'  {label:<6}: {mean_tp[tp_name]:.4f}')
print('=' * 50)

## 셀 6: 리더보드 비교 시각화

In [ ]:
leaderboard = {
    'BEVFusion\n(SOTA)':     {'NDS': 0.713, 'mAP': 0.680},
    'CenterPoint\n(Pillar)': {'NDS': 0.598, 'mAP': 0.503},
    'BEVDet\n(R50)':         {'NDS': 0.482, 'mAP': 0.398},
    'FCOS3D\n(camera)':      {'NDS': 0.428, 'mAP': 0.343},
    'LSS\n(우리 구현)':       {'NDS': NDS,   'mAP': mAP},
}

names  = list(leaderboard.keys())
nds_v  = [leaderboard[n]['NDS'] for n in names]
map_v  = [leaderboard[n]['mAP'] for n in names]
colors = ['#3498DB','#9B59B6','#2ECC71','#E67E22','#E74C3C']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('nuScenes Detection Leaderboard 비교', fontsize=14, fontweight='bold')

for ax, vals, title, ylabel in [
    (axes[0], nds_v, 'NDS (nuScenes Detection Score)', 'NDS'),
    (axes[1], map_v, 'mAP (Mean Average Precision)',   'mAP'),
]:
    bars = ax.bar(names, vals, color=colors, edgecolor='white')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1.0)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('submission_output/leaderboard_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# 클래스별 AP 레이더 차트
fig, ax = plt.subplots(figsize=(8,8), subplot_kw={'projection': 'polar'})
cls_names = list(ap_results.keys())
aps = [ap_results[c] for c in cls_names]
N = len(cls_names)
angles = [n/float(N)*2*np.pi for n in range(N)] + [0]
aps_plot = aps + aps[:1]
ax.plot(angles, aps_plot, 'o-', linewidth=2, color='#E74C3C')
ax.fill(angles, aps_plot, alpha=0.25, color='#E74C3C')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(cls_names, size=9)
ax.set_ylim(0, 1)
ax.set_title('클래스별 AP - LSS BEV 모델', pad=20)
plt.tight_layout()
plt.savefig('submission_output/class_ap_radar.png', dpi=150, bbox_inches='tight')
plt.show()

## 셀 7: 실제 nuScenes mini 연동 가이드

In [ ]:
print('=' * 65)
print('  실제 nuScenes mini 데이터로 전환하는 방법')
print('=' * 65)
print()
print('1. 데이터 다운로드 (회원가입 필요):')
print('   https://www.nuscenes.org/nuscenes')
print('   → nuScenes mini (4GB) 다운로드')
print('   → data/nuscenes/ 에 압축 해제')
print()
print('2. NuScenes 객체 로드:')
print('''
from nuscenes.nuscenes import NuScenes
nusc = NuScenes(version="v1.0-mini",
                dataroot="data/nuscenes",
                verbose=True)
''')
print('3. 공식 평가 실행:')
print('''
from nuscenes.eval.detection.evaluate import NuScenesEval
nusc_eval = NuScenesEval(
    nusc,
    config=cfg,
    result_path="submission_output/results_nusc.json",
    eval_set="mini_val",
    output_dir="submission_output/eval_results",
    verbose=True
)
metrics, _ = nusc_eval.evaluate()
print(f"NDS: {metrics.nd_score:.4f}")
print(f"mAP: {metrics.mean_ap:.4f}")
''')
print('4. 리더보드 제출:')
print('   nuscenes.org/object-detection → Submit')
print('   results_nusc.json 업로드')
print()
print(f'현재 파이프라인 상태:')
print(f'  제출 파일  : submission_output/results_nusc.json')
print(f'  스키마 검증: 통과')
print(f'  Mock NDS   : {NDS:.4f}')
print(f'  Mock mAP   : {mAP:.4f}')
print('=' * 65)